**Машинное обучение в экономике**

**Семинар 7. Нейросети**

Установка библиотек

In [ ]:
# !python.exe -m pip install --upgrade pip
# !pip install numpy
# !pip install pandas
# !pip install scikit-learn
# !pip install openpyxl

In [ ]:
import numpy as np                                        # базовые операции с массивами
import pandas as pd                                       # базовые операции с датафреймами
from sklearn.model_selection import train_test_split      # разделение выборки
from copy import deepcopy
import matplotlib.pyplot as plt                           # графики
import scipy                                              # распределения
import math
import torch                                              # нейросети
from torch import nn
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression       # логистическая регрессия
from sklearn.preprocessing import scale                   # нормализация
import sklearn
from torch.utils.data import Dataset, DataLoader          # работа с данными

**Генерация данных** 🐰

In [ ]:
# Для воспроизводимости
np.random.seed(777)

# Число наблюдений
n = 20000

# Названия переменных
features_names = ["age", "educ", "married", "nchildren", "work", "income"]

# Число признаков
m = len(features_names)

# Ковариации
sigma = np.empty(shape=(m, m), dtype = 'object')
sigma = pd.DataFrame(np.ones(shape=(m, m), dtype = 'object'),
                     index = features_names,
                     columns = features_names)
  # age
sigma.loc["age", "educ"] = 0.3
sigma.loc["age", "married"] = 0.4
sigma.loc["age", "nchildren"] = 0.5
sigma.loc["age", "work"] = 0.3
sigma.loc["age", "income"] = 0.4
  # educ
sigma.loc["educ", "married"] = 0.3
sigma.loc["educ", "nchildren"] = -0.1
sigma.loc["educ", "work"] = 0.5
sigma.loc["educ", "income"] = 0.6
  # married
sigma.loc["married", "nchildren"] = 0.5
sigma.loc["married", "work"] = 0.3
sigma.loc["married", "income"] = 0.2
  # nchildren
sigma.loc["nchildren", "work"] = 0.1
sigma.loc["nchildren", "income"] = 0.1
  # work
sigma.loc["work", "income"] = 0.7

# Делаем нижнюю и верхнюю части ковариационной
# матрицы симметричными
for i in range(0, m):
  for j in range(0, i):
    sigma.iloc[i, j] = sigma.iloc[j, i]

# Генирируем данные из многомерного нормального распределения
df = np.random.multivariate_normal(mean = np.zeros(m), cov = sigma, size = n)
df = pd.DataFrame(df)
df.columns = features_names

# Приводим данные к реалистичному виду
  # age
df["age"] = pd.cut(df.loc[:, "age"],
                   bins = scipy.stats.norm.ppf
                     (
                     np.linspace(start = 0, stop = 1, num = 42),
                                 loc = 0, scale = 1
                     ),
                   labels = range(20, 61)
                  )
  # educ
df["educ"] = pd.cut(df.loc[:, "educ"],
                    bins = scipy.stats.norm.ppf
                      (
                        [0, 0.6, 0.9, 1],
                        loc = 0, scale = 1
                      ),
                      labels = range(1, 4)
                   )
  # married
df["married"] = pd.cut(df.loc[:, "married"],
                       bins = scipy.stats.norm.ppf
                         (
                           [0, 0.4, 1],
                           loc = 0, scale = 1
                         ),
                       labels = range(0, 2)
                      )
  # nchildren
df["nchildren"] = pd.cut(df.loc[:, "nchildren"],
                         bins = scipy.stats.norm.ppf(
                           np.concatenate
                             (
                               (
                                 [0],
                                 scipy.stats.poisson.cdf(k = range(0, 10),
                                                         mu = 2),
                                 [1]
                               )
                             ),
                           loc = 0, scale=  1),
                           labels = range(0, 11)
                        )
  # work
df["work"] = pd.cut(df.loc[:, "work"],
                    bins = scipy.stats.norm.ppf
                             (
                              [0, 0.3, 1],
                              loc = 0, scale = 1
                             ),
                    labels = range(0, 2)
                    )
  # income
df["income"] = np.round(
                         scipy.stats.expon.ppf
                           (
                             scipy.stats.norm.cdf(df["income"]), scale = 10
                           )
                       ) + 10

# Конвертируем данные в нужный формат
df = df.astype(float)

# Симулируем покупки
buy_li = (
           2.15 -
           0.03 * df["age"] +
           0.00005 * df["age"] ** 2 -
           0.6 * (df["educ"] == 2.0) -
           0.8 * (df["educ"] == 3.0) +
           0.5 * df["married"] +
           0.2 * df["nchildren"] -
           0.6 * df["work"] -
           0.5 * np.log(df["income"]) +
           0.02 * np.log(df["income"]) * df["nchildren"] +
           0.02 * df["age"] * df["married"] -
           0.01 * df["age"] * df["nchildren"] +
           0.01 * df["age"] * df["work"] -
           0.001 * df["age"] * df["income"] +
           0.03 * df["income"] * df["married"] -
           0.02 * df["income"] * df["educ"] +
           0.04 * df["income"] * df["educ"] * df["work"]
          )
buy_prob = scipy.stats.t.cdf(buy_li * 2, df = 5)
df["buy"] = np.random.binomial(n = 1, p = buy_prob, size = n)

# Доля покупок
print({'buy': np.mean(df["buy"])})

# Корреляционная матрица
print(df.astype(float).corr(method = 'pearson'))

# Данные
print(df)

**Тензоры** 🐱

В библиотеке `PyTorch` данные представлены в форме тензоров, которые удобно представлять как обобщение матриц на многомерный случай.

In [ ]:
# Скаляр (тензор размерности 0)
tens_scalar = torch.tensor(8)
print('Скаляр - тензор размерности 0')
print(tens_scalar)

# Вектор (тензор размерности 1)
tens_vector = torch.tensor([8, 5])
print('Вектор - тензор размерности - 1')
print(tens_vector)

# Матрица (тензор размерности 2)
print('Матрица - тензор размерности - 2')
tens_matrix = torch.tensor([
                            [
                             [1, 2, 3],
                             [4, 5, 6],
                             [7, 8, 9]
                            ]
                           ])
print(tens_matrix )

# Тензор размерности 3
print('Тензор размерности 3')
tens_3d = torch.tensor([
                        [[1, 2],
                         [3, 4]
                        ],
                        [[5, 6],
                         [7, 8]
                        ]
                       ])
print(tens_3d)

По аналогии с тем, как нас может интересовать не вся матрица, а ее столбцы или строки (векторы), в случае с тензором мы можем также выбрать лишь часть его элементов, относящихся к тому или иному измерению.

Синтаксис `tens_3d[i, j, k]` возвращает `k`-й элемент `j`-й строки `i`-й матрицы трехмерного тензора `tens_3d`.

Если бы у нас был четырехмерный тензор, то синтаксис `tens_4d[i, j, k, t]` возвращал бы `t`-й элемент `k`-й строки `j`-й матрицы `t`-го трехмерного тензора у четырехмерного тензора `tens_4d`.

In [ ]:
# Весь тензор
print('Тензор')
print(tens_3d)

# Достанем матрицы из трехмерного тензора (двумерные тензоы)
  # первая матрица
print('Первая матрица')
print(tens_3d[0])
  # вторая матрица
print('Вторая матрица')
print(tens_3d[1])

# Достанем строки матриц (одномерные тензоры - векторы)
  # первая строка первой матрицы
print('Первая строка первой матрицы')
print(tens_3d[0, 0])
  # вторая строка первой матрицы
print('Вторая строка первой матрицы')
print(tens_3d[0, 1])
   # вторая строка второй матрицы
print('Вторая строка второй матрицы')
print(tens_3d[1, 1])

# Достанем элементы, находящихся во второй
# строке первого столбца второй матрицы
print('Вторая матрица, вторая строка, первый элемент')
print(tens_3d[1, 1, 0])

# Достанем вторую строку из каждой матрицы
print('Вторая строка каждой матрицы')
print(tens_3d[:, 1, :])

# Достанем первый столбец из каждой матрицы
print('Первый столбец каждой матрицы')
print(print(tens_3d[:, :, 1]))

Создание особых видов тензоров

In [ ]:
# Тензор из случайных чисел
print('Тензор из случайных чисел')
tens_rand = torch.rand(size=(2,            # число матриц
                             3,            # число строк
                             4))           # число столбцов
print(tens_rand)

# Тензор из нулей
print('Тензор из нулей')
tens_zero = torch.zeros(size = (4,         # число матриц
                                3,         # число строк
                                2))        # число столбцов
print(tens_zero)

# Тензор из единиц
print('Четырехмерный тензор из единиц')
tens_ones = torch.ones(size = (2,          # число тензоров
                               3,          # число матриц
                               4,          # число строк
                               5))         # число столбцов
print(tens_ones)

Информация о тензоре

In [ ]:
# Размерность тензора
print('Число измерений тензора')
print(tens_3d.ndim)

# Тип данных тензора
print('Тип данных тензора')
print(tens_3d.dtype)

# Тензор обрабаывается ресурсами процессора CPU или видеокарты GPU
print('Процессор или видеокарта')
print(tens_3d.device)

**Важно** - для того, чтобы проводить операции над тензорами, они должны иметь подходящие размерности (по аналогии с матрицами) `ndim`, а также одинаковый тип, то есть `dtype` и` device`.

In [ ]:
# Двумерные тензоры (матрицы)
  # первый тензор
tens1 = torch.rand(size   = (2, 3),        # размерность
                   dtype  = torch.float32, # тип данных
                   device = 'cpu')         # СPU (процессор) или GPU (видеокарта)
print('Первый тензор')
print(tens1)
  # второй тензор
tens2 = torch.rand(size   = (3, 2),        # размерность
                   dtype  = torch.float32, # тип данных
                   device = 'cpu')         # СPU (процессор) или GPU (видеокарта)
print('Второй тензор')
print(tens2)

# Произведение тензоров
tens_prod = torch.matmul(tens1, tens2)
print('Произведение тензоров')
print(tens_prod)

Если у вас имеется достаточно мощная видеокарта, то быстрее обучать нейросети с использованием ресурсов GPU (видеокарта), чем CPU (процессора).

Задачи для самоподготовки 🐻

1.   Посмотрите на ошибку, возникающую в случае, когда тензоры имеют различные значения аргументов `dtype` и `device`. 🥉

**Загрузка и первичный анализ данных** 🐱

In [ ]:
# Разделим целевую переменную и признаки
target   = df.loc[:, ['buy']]                   # целевая переменная
features = df.loc[:, df.columns.drop('buy')]    # матрица признаков
target   = np.squeeze(target)                   # преобразуем из вектора столбца
                                                # в одномерный массив

# Разделим выборку на обучающую и тестовую
features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size = 0.2, random_state = 777)

# Сохраним число наблюдений обучающей и тестовой выборок
n_train = len(target_train)
n_test  = len(target_test)

# Вернем исходную сортировку индексов
features_train = features_train.reset_index(drop = True)
target_train   = target_train.reset_index(drop = True)
features_test  = features_test.reset_index(drop = True)
target_test    = target_test.reset_index(drop = True)

In [ ]:
# Подключаем GPU, если есть такая возможность
device = "cuda" if torch.cuda.is_available() else "cpu"

# Проверяем, что именно подключилось в нашем случае
print(device)

**Очень важно** - при обучении нейросетей критически важно нормализовать данные, поскольку в противном случае градиентный спуск может работать чрезвычайно плохо.

In [ ]:
# Подготовим объект, осуществляющий нормализацию
scaler = sklearn.preprocessing.StandardScaler().set_output(transform = "pandas").fit(features_train)

# Нормализуем данные
features_train = scaler.transform(features_train)  # обучающая выборка
features_test  = scaler.transform(features_test)   # тестовая выборка

In [ ]:
# Сохраним признаки и целевую переменную в тензорном формате
  # обучающая выборка
x_train = torch.tensor(features_train.values.astype(np.float32)).to(device)
y_train = torch.from_numpy(target_train.values.astype(np.float32)).to(device)
  # тестовая выборка
x_test = torch.tensor(features_test.values.astype(np.float32)).to(device)
y_test = torch.from_numpy(target_test.values.astype(np.float32)).to(device)

#
print('Признаки до превращения в тензор')
print(features_train[0:10])

# Посмотрим на признаки
print('Признаки')
print(x_train[0:10])

# Изучим целевую переменную
print('Целевая переменная')
print(y_train[0:10])

**Логистическая регрессия как частный случай нейросети** 🐱

**Общий алгоритм**

1. Написать класс, описывающий структуру нейросети: количество скрытых слоев и нейронов в них, функции активации и т.д.

2. Выбрать функцию потерь, например, логистическую.

3. Выбрать оптимизатор, например, градиентный спуск.

4. Запустить процесс обучения нейросети: минимизировать функцию потерь по параметрам нейросети (весам) с помощью выбранного оптимизатора.

Техническое примечание ⚡

Модели в PyTorch создаются как классы, являющиеся наследниками класса `nn.Module`. Код `super().__init__()` вызывает конструктор родительского класса `nn.Module`, который затем мы дополняем собственным кодом.

Создаваемый класс должен включать, во-первых, конструктор `__init__()`. В нем обычно описывается структура нейросети: функции активации, число слоев и нейронов в них и т.д.

Во-вторых, класс должен включать метод `forward()`, в котором описывается, как именно нейросеть рассчитывает значения, в дальнейшем подаваемые в функцию потерь: условные вероятности, условные математические ожидания и т.д.

Каждый слой нейросети включает `out_features` нейронов, в которых рассчитываются значения с помощью информации, полученной от `in_features` нейронов предыдущего слоя. Функция `nn.Linear()` рассчитывает `out_features` линейных комбинаций из `in_features` нейронов. Если `bias = True`, то линейные комбинации рассчитываются с константой.

Функция `nn.Sigmoid()` позволяет применить сигмоидную функцию активации (функция распределения стандартного логистического распределения), например, к посчитанным линейным комбинациям.

Специальный аргумент `self` отражает экземпляр класса.

In [ ]:
# Логистическая регрессия как нейросеть
class MyLogit(nn.Module):
    # Конструктор
    def __init__(self, n_features):
        super().__init__()

        # Подготавливаем единственный скрытый слой
          # линейная комбинация
        self.lincomb_1 = nn.Linear(in_features  = n_features,
                                   out_features = 1,
                                   bias         = True)
          # функция активации
        self.activation_1 = nn.Sigmoid()

    # Метод для получения прогнозов, где 'x' отражает информацию
    # входного слоя, то есть признаки
    def forward(self, x):
        # Применяем функцию активации к линейной комбинации
        val = self.activation_1(self.lincomb_1(x))
        return val

Техническое примечание ⚡

Для того, чтобы обучить и использовать нейросеть, необходимо создать экземпляр нашего класса. Аргумент `n_features` отвечает за количество нейронов входного слоя, то есть равняется числу признаков. Код `to(device) `необходим для того, чтобы указать, будет ли наша модель обучаться с использованием ресурсов процессора или видеокарты.

In [ ]:
# Создадим модель как экземпляр нашего класса
logit = MyLogit(n_features = x_train.size(dim = 1)).to(device)

Техническое примечание ⚡

Чтобы обучить нейронную сеть, необходимо выбрать функцию потерь. Ее можно либо запрограммировать самостоятельно, либо выбрать из списка, указанного в [документации](https://pytorch.org/docs/stable/nn.html#loss-functions). Например, `nn.BCELoss()` отражает логистическую функцию потерь.

In [ ]:
# Выбираем логистическую функцию потерь
logit_loss = nn.BCELoss()

Веса и смещения, относящиеся к линейным комбинациям, хранятся в виде их свойств `weiights.data` и `bias.data` соответственно.

In [ ]:
# Изучим изначальные, случайным образом заданные веса
  # Смещение
print('Смещение')
print(logit.lincomb_1.bias.data)
  # Веса
print('Веса')
print(logit.lincomb_1.weight.data)

In [ ]:
# Начальные веса можно изменять, например, обнулить
logit.lincomb_1.weight.data.fill_(0.0)
logit.lincomb_1.bias.data.fill_(0.0)

Техническое примечание ⚡

Чтобы получить прогноз с использованием текущих весов нейросети, необходимо применить метод `forward()`. При этом функция `squeeze()` убирает лишние размерности из тензора, в частности, превращая матрицу размерности `n x 1` в вектор размера `n`.

In [ ]:
# Начальные прогнозы
logit_pred_train = logit.forward(x_train).squeeze()
print(logit_pred_train[0:10])

Техническое примечание ⚡

Метод численной оптимизации можно выбрать из списка, указанного в [документации](https://pytorch.org/docs/stable/optim.html). Например, оптимизатор `torch.optim.SGD()` реализует градиентный спуск для обучения параметров `params` со скоростью обучения `lr`. Для превращения его в стохастический необходимы дополнительные манипуляции, которые мы обсудим позже.

In [ ]:
# Выбираем градиентный спуск в качестве оптимизатора
logit_opt = torch.optim.SGD(params = logit.parameters(), # параметры модели
                            lr     = 1)                  # скорость обучения

Техническое примечание ⚡

Обучение параметров нейросети (весов) происходит с помощью цикла, в котором каждую итерацию необходимо:

1. Из технический соображений перевести модель в обучающий режим с помощью метода `train()`.
2.   Посчитать прогнозы нейросети с текущими параметрами (обычно весами) с помощью метода `forward()`.
3.   Рассчитать значение функции потерь, передав в нее посчитанные на предыдущем шаге значения.
4.   Посчитать градиент функции потерь по параметрам нейросети с помощью метода обратного распространения ошибки (backpropagation), вызвав у функции потерь метод `backward()`. Из технических соображений этот градиент необходимо предварительно обнулить, вызвав у оптимизатора метода `zero_grad()`.
5. Используя посчитанный градиент совершить шаг алгоритма численной оптимизации, вызвав у оптимизатора метод `step()`.


In [ ]:
# Количество итераций
n_iter = 100

# Обучение и тестирование моделей
# Примечание - часто вместо iter используют название epoch
for iter in range(n_iter):

    # Устанавливаем обучающий режим модели
    logit.train()

    # Считаем прогнозы нейросети (значения выходного слоя)
    logit_prob_train = logit.forward(x_train).squeeze()         # вероятности

    # Считаем функцию потерь и точность
    logit_loss_train = logit_loss(logit_prob_train, y_train)

    # Обнуляем посчитанные ранее градиенты функции потерь
    logit_opt.zero_grad()

    # Дифференцируем функцию потерь по весам методом обратного
    # распространения ошибки (backpropagation)
    logit_loss_train.backward()

    # Совершаем шаг алгоритма численной оптимизации (обновляем веса)
    logit_opt.step()

    # Предварительные результаты
    print(f"Итерация {iter}: Функция потерь = {logit_loss_train:.5f}, " + \
          f"Веса = {logit.lincomb_1.weight.data.squeeze().numpy()}")

In [ ]:
# Сравним результаты с sclearn
logit2 = LogisticRegression(penalty       = None,
                            solver        = 'lbfgs',
                            fit_intercept = True)
logit2.fit(features_train, target_train)
coef_tbl = pd.DataFrame({'Веса':         features.columns,
                         'sclearn':      logit2.coef_.flatten(),
                         'PyTorch':      logit.lincomb_1.weight.data.squeeze().detach().cpu().numpy()})
print(coef_tbl)

**Нейронная сеть с двумя скрытыми слоями и исключением (dropout)** 🐱

Техническое примечание ⚡

Чтобы добавить исключение, необходимо воспользоваться функцией `nn.Dropout`(), у которой аргумент `p` отвечает за вероятность отключения нейрона. Эту функцию можно применять отдельно к каждому слою с различными параметрами.

**Рекомендация** - на практике класс нейросети может включать множество дополнительных удобных для вас методов. Например, вы можете добавить в него методы для расчета некоторой метрики качества, прогнозов и т.д.

Для удобства включим в конструктор аргументы `n_layer1 `и `n_layer2`, отвечающие за число нейронов в каждом слое.

In [ ]:
# Нейросеть
class MyNN(nn.Module):
    # Конструктор
    def __init__(self, n_features, n_layer1 = 30, n_layer2 = 20):
        super().__init__()

        # Подготавливаем первый слой
        self.lincomb_1 = nn.Linear(in_features  = n_features,  # линейная комбинация
                                   out_features = n_layer1,
                                   bias         = True)
        self.activation_1 = nn.Tanh()                         # функция активации
        self.dropout_1    = nn.Dropout(p = 0.2)               # исключение

        # Подготавливаем второй слой
        self.lincomb_2 = nn.Linear(in_features  = n_layer1,   # линейная комбинация
                                   out_features = n_layer2,
                                   bias         = True)
        self.activation_2 = nn.ELU()                          # функция активации
        self.dropout_2    = nn.Dropout(p = 0.3)               # исключение

        # Подготавливаем выходной слой
        self.lincomb_out = nn.Linear(in_features  = n_layer2, # линейная комбинация
                                     out_features = 1,
                                     bias         = True)
        self.activation_out = nn.Sigmoid()                    # функция активации

    # Метод для получения прогнозов, где x отражает информацию
    # входного слоя, то есть признаки
    def forward(self, x):
        # Считаем значения первого слоя
        val = self.dropout_1(self.activation_1(self.lincomb_1(x)))

        # Считаем значения второго слоя
        val = self.dropout_2(self.activation_2(self.lincomb_2(val)))

        # Считаем значение выходного слоя
        val = self.activation_out(self.lincomb_out(val))

        # Возвращаем результат
        return val

In [ ]:
# Создадим модель как экземпляр нашего класса
mynn = MyNN(n_features = x_train.size(dim = 1)).to(device)

In [ ]:
# Выбираем логистическую функцию потерь
mynn_loss = nn.BCELoss()

In [ ]:
# Начальные прогнозы (оценки условных вероятностей)
mynn_pred_train = mynn.forward(x_train).squeeze()
print(mynn_pred_train[0:10])

In [ ]:
# Выбираем градиентный спуск в качестве оптимизатора
mynn_opt = torch.optim.SGD(params = mynn.parameters(),   # параметры модели
                           lr     = 0.5)                 # скорость обучения

Техническое примечание ⚡

На протяжении оптимизации параметров нейросети часто бывает удобно следить за тем, насколько хорошо она работает на валидационной выборке (для краткости вместо нее мы используем тестовую). В частности, если в какой-то момент функция потерь на валидационной выборке начинает расти, то это может свидетельствовать в пользу переобучения.

Для того, чтобы переключить модель в режим тестирования, необходимо вызвать метод `eval()` и производить прогнозы внутри блока `torch.inference_mode()`. В частности, это гарантирует, что при использовании исключения (dropout) прогнозы будут строиться по полной, а не утонченной нейросети.

In [ ]:
# Количество итераций
n_iter = 1000

# Обучение и тестирование моделей
for iter in range(n_iter):

    # Устанавливаем обучающий режим модели
    mynn.train()

    # Считаем прогнозы нейросети
    mynn_prob_train = mynn.forward(x_train).squeeze()

    # Считаем функцию потерь и точность
    mynn_loss_train = mynn_loss(mynn_prob_train, y_train)

    # Обнуляем посчитанные ранее градиенты функции потерь
    mynn_opt.zero_grad()

    # Дифференцируем функцию потерь по весам методом обратного
    # распространения ошибки (backpropagation)
    mynn_loss_train.backward()

    # Совершаем шаг алгоритма численной оптимизации (обновляем веса)
    mynn_opt.step()

    # Устанавливаем тестирующий режим модели
    mynn.eval()
    with torch.inference_mode():
      # Считаем прогнозы нейросети
      mynn_prob_test = mynn.forward(x_test).squeeze()

      # Считаем функцию потерь и точность
      mynn_loss_test = mynn_loss(mynn_prob_test, y_test)


    # Предварительные результаты
    if iter % 50 == 0:
      print(f"Итерация {iter}: Функция потерь (train) = {mynn_loss_train:.5f} | " +
            f"Функция потерь (test) = {mynn_loss_test:.5f} ")

In [ ]:
# Оценим условные вероятности обеими моделями на тестовой выборке
logit_prob_test = logit.forward(x_test).squeeze()
mynn_prob_test  = mynn.forward(x_test).squeeze()

In [ ]:
# Получим прогнозы по обеим моделям на тестовой выборке
logit_pred_test = (logit_prob_test >= 0.5).to(int)
mynn_pred_test  = (mynn_prob_test >= 0.5).to(int)

In [ ]:
# Сравним точность прогнозов на тестовой выборке
logit_acc_test = (logit_pred_test == y_test).sum().item() / len(y_test)
mynn_acc_test  = (mynn_pred_test == y_test).sum().item() / len(y_test)
print(f"ACC логит = {logit_acc_test}, ACC нейросеть = {mynn_acc_test}")

**Продвинутая работа с данными** 🐱

Технический комментарий ⚡

Для того, чтобы упростить работу с данными, в библиотеке PyTorch имееются сепциальные классы `Dataset` и `DataLoader`. В частности, благодаря ним можно реализовать **мини-пакетный** градиентный спуск.

Сперва необходимо сформировать собственный класс для работы с данными, являющийся наследником класса `Dataset`. Этот класс обязательно должен включать конструктор `__init__()`, метод доступа к данным `__getitem__()`, а также метод, возвращающий размер данных `__len__()`.

In [ ]:
# Класс для работы с данными
class MyData(Dataset):
  # Конструктор
  def __init__(self, x, y):
    self.x = x                      # признаки
    self.y = y                      # целевая переменная
    self.len = self.x.shape[0]      # число наблюдений
  # Метод, возвращающий данные:
  # строки index из x и y
  def __getitem__(self, index):
      return self.x[index], self.y[index]
  # Метод, возвращающий число наблюдений
  def __len__(self):
      return self.len

In [ ]:
# Сформируем данные
data_train = MyData(x = x_train, y = y_train)

In [ ]:
# Посмотрим на признаки обучающей выборки
print(data_train.x)

In [ ]:
# Достанем первые несколько строк данных, то есть
# сразу и признаков, и целевой переменной
print(data_train[0:10])

Техническое примечание ⚡

Функция `DataLoader()` может быть использована для того, чтобы разбить наши данные `dataset` на мини-пакеты размера `batch_size`. Если `shuffle = True`, то после того, как перебираются все возможные мини-пакеты, они формируются заново случайным образом, что делает алгоритм оптимизации стохастическим.

In [ ]:
# Специальный объект для удобного доступа к данным
loader_train = DataLoader(dataset    = data_train, # данные
                          batch_size = 200,        # размер каждого мини-пакета
                          shuffle    = True)       # перетосовка данных

Техническое примечание ⚡

С помощью кода `x_loader, y_loader in loader_train` мы можем перебрать все мини-пакеты внутри наших данных. Таким образом, для обучения нейросети мы будем использовать тот же код, что и раньше, но вместо исходных данных использовать мини-пакеты `x_loader, y_loader` из наших данных `loader_train`.

In [ ]:
# Количество итераций
n_iter = 10

# Обучение и тестирование моделей
for iter in range(n_iter):                   # итерация
    for x_loader, y_loader in loader_train:  # подитерации с мини-пакетами

      # Устанавливаем обучающий режим модели
      mynn.train()

      # Считаем прогнозы нейросети
      mynn_prob_train = mynn.forward(x_loader).squeeze()

      # Считаем функцию потерь и точность
      mynn_loss_train = mynn_loss(mynn_prob_train, y_loader)

      # Обнуляем посчитанные ранее градиенты функции потерь
      mynn_opt.zero_grad()

      # Дифференцируем функцию потерь по весам методом обратного
      # распространения ошибки (backpropagation)
      mynn_loss_train.backward()

      # Совершаем шаг алгоритма численной оптимизации (обновляем веса)
      mynn_opt.step()

    # Устанавливаем тестирующий режим модели
    mynn.eval()
    with torch.inference_mode():
      # Считаем прогнозы нейросети
      mynn_prob_test = mynn.forward(x_test).squeeze()

      # Считаем функцию потерь и точность
      mynn_loss_test = mynn_loss(mynn_prob_test, y_test)

    # Предварительные результаты
    print(f"Итерация {iter}: Функция потерь (train) = {mynn_loss_train:.5f} | " +
          f"Функция потерь (test) = {mynn_loss_test:.5f} ")

In [ ]:
# Сравним точность прогнозов на тестовой выборке
mynn_prob_test2 = mynn.forward(x_test).squeeze()
mynn_pred_test2 = (mynn_prob_test2 >= 0.5).to(int)
mynn_acc_test2  = (mynn_pred_test2 == y_test).sum().item() / len(y_test)
print(logit_acc_test, mynn_acc_test , mynn_acc_test2)

**Ручной расчет** 🐱

Рассмотрим нейросеть с двумя признаками, одним скрытым слоем с сигмоидной функцией активации и выходным слоем, который также имеет сигмоидную функцию активации.

**Важно** - по мере выполнения этого задания сделайте (например, на листочке) графическое изображение нейросети с подписанными весами, функцией активации и т.д.

In [ ]:
# Наша нейросеть
class MyNN2(nn.Module):
    # Конструктор
    def __init__(self, n_features):
        super().__init__()

        # Подготавливаем единственный скрытый слой
        self.lincomb_1 = nn.Linear(in_features  = n_features,
                                   out_features = 2,
                                   bias         = False)
        self.activation_1 = nn.Sigmoid()

        # Выходной слой
        self.lincomb_out = nn.Linear(in_features  = 2,
                                     out_features = 1,
                                     bias         = False)
        self.activation_out = nn.Sigmoid()

    # Метод для получения прогнозов, где x отражает информацию
    # входного слоя, то есть признаки
    def forward(self, x):
        # Применяем функцию активации к линейной комбинации
        val = self.activation_1(self.lincomb_1(x))
        val = self.activation_out(self.lincomb_out(val))
        return val

Для простоты представим, что у нас имеется лишь одно наблюдение, причем первый признак равен $X_{1}=1$, второй признак равен $X_{2}=-2$, а значение целевой переменной $Y = 1$.

In [ ]:
# Данные
x = np.array([1, -2])
y = np.array([1])

Запрограммируем сигмоидную функцию:

$$h(t) = \frac{1}{1+e^{-t}}$$

In [ ]:
# Сигмоида
def sigmoid(x):
  val = 1 / (1 + np.exp(-x))
  return(val)

Запрограммируем производную сигмоидной функции:

$$h'(t) = \frac{e^{-t}}{(1+e^{-t})^2}=h(t)(1-h(t))$$

In [ ]:
# Производная сигмоиды (короткая формула)
def dsigmoid(x):
  val = sigmoid(x)
  val = val * (1 - val)
  return(val)

Предположим следующие веса:

$$\omega_{11} = (0.1, 0.2)\qquad \omega_{12} = (0.3, 0.4)\qquad\text{первый слой}$$

$$\omega_{21} = (0.5, 0.6)\qquad \text{веса, с которыми нейроны первого слоя заходят в выходной слой}$$

In [ ]:
# Веса
w11 = np.array([0.1, 0.2])
w12 = np.array([0.3, 0.4])
w21 = np.array([0.5, 0.6])

Таким образом, в первом слое рассчитываются следующие линейные комбинации:

$$s_{11} = \omega_{111}X_{1} + \omega_{112}X_{2}$$

$$s_{12} = \omega_{121}X_{1} + \omega_{122}X_{2}$$

In [ ]:
# Линейные комбинации первого слоя
s11 = w11[0] * x[0]  + w11[1] * x[1]
s12 = w12[0] * x[0]  + w12[1] * x[1]
print(s11, s12)

Применение функции активации:

$$h_{1} = h(s_{1})\qquad h_{2} = h(s_{2})$$

In [ ]:
# Функция активации от второго слоя
h1 = sigmoid(s11)
h2 = sigmoid(s12)
print(h1, h2)

Линейная комбинация функций активации:

$$s_{21} = \omega_{211}h_{1} + \omega_{212}h_{2}$$

In [ ]:
# Линейная комбинация второго слоя
s21 = w21[0] * h1 + w21[1] * h2
print(s21)

Значение выходного слоя нейросети:

$$\text{output} = \hat{P}(y = 1 | x) = h(s_{21})$$

In [ ]:
# Выходное значение слоя
output = sigmoid(s21)
print(output)

Запрограммируем логистическую функцию потерь

In [ ]:
# Логистическая функция потерь
def log_loss(Y, F):
  val = np.mean(-Y * np.log(F) - (1 - Y) * np.log(1 - F))
  return(val)

Запрограммируем производную логистической функции потерь

In [ ]:
# Производная функции потерь
def d_log_loss(Y, F):
  val = np.mean(-Y / F + (1 - Y) / (1 - F))
  return(val)

Рассчитаем градиент в несколько шагов

$$\frac{\partial \text{ loss}}{\partial \text{ output}} = \text{loss}'(\text{output})$$

In [ ]:
d_log_loss_d_output = d_log_loss(y, output)
print(d_log_loss_d_output)

$$\frac{\partial \text{ output}}{\partial s_{21}} = h'(s_{21})$$

In [ ]:
d_output_d_s21 = dsigmoid(s21)
print(d_output_d_s21)

$$\frac{\partial s_{21}}{\partial h_{1}} = \omega_{211}$$

In [ ]:
d_s21_d_h1 = w21[0]
print(d_s21_d_h1)

$$\frac{\partial h_{1}}{\partial s_{11}} = h'(s_{11})$$

In [ ]:
d_h1_d_s11 = dsigmoid(s11)
print(d_h1_d_s11)

$$\frac{\partial s_{11}}{\partial \omega_{111}} = X_{1}$$

In [ ]:
d_s11_d_w111 = x[0]
print(d_s11_d_w111)

Используем алгоритм обратного распространения ошибки (backpropagation) для расчета производной:

$$\frac{\partial \text{ loss}}{\partial \omega_{111}} = \frac{\partial \text{ loss}}{\partial \text{ output}}\frac{\partial \text{ output}}{\partial s_{21}}\frac{\partial s_{21}}{\partial h_{1}}\frac{\partial h_{1}}{\partial s_{11}}\frac{\partial s_{11}}{\partial \omega_{111}}$$

In [ ]:
d_log_loss_d_w111 = np.mean((d_log_loss_d_output * d_output_d_s21 * d_s21_d_h1 *
                             d_h1_d_s11 * d_s11_d_w111))
print(d_log_loss_d_w111)

Положим скорость обучения $\alpha = 0.1$ и обновим вес $\omega_{111}$ сделав один шаг алгоритма градиентного спуска:

$$\omega_{111}^{\text{new}} = \omega_{111} - \alpha\times \frac{\partial \text{ loss}}{\partial \omega_{111}}$$

In [ ]:
# Скорость обучения
lr =  0.1

# Шаг алгоритма градиентго спуска
w111_new = w11[0] - lr * d_log_loss_d_w111

# Сравнение исходного и полученного значений
print(f"Было w111 = {w11[0]}, Стало w111 = {w111_new}")

Если имеется более, чем одно наблюдение, то итоговый градиент считается как сумма градиентов по всем наблюдениям.

Проверим верность проведенных расчетов сопоставив результаты с `PyTorch`.

In [ ]:
# Подготовим данные
x_train = torch.tensor(x.astype(np.float32)).to('cpu')
x_train = torch.reshape(x_train, [1, len(x)])
y_train = torch.from_numpy(y.astype(np.float32)).to('cpu')
print(x_train)

In [ ]:
# Создадим модель как экземпляр нашего класса
mynn2 = MyNN2(n_features = x_train.size(dim = 1)).to('cpu')

In [ ]:
# Выбираем логистическую функцию потерь
mynn2_loss = nn.BCELoss()

In [ ]:
# Установим веса
mynn2.lincomb_1.weight.data[0]   = torch.tensor(w11)
mynn2.lincomb_1.weight.data[1]   = torch.tensor(w12)
mynn2.lincomb_out.weight.data[0] = torch.tensor(w21)

# Посмотрим на веса, помощенные в модель
print(mynn2.lincomb_1.weight.data)
print(mynn2.lincomb_out.weight.data)

In [ ]:
# Сопоставим прогнозы (значения выходного слоя)
mynn2_prob = mynn2.forward(x_train).squeeze()
print(f" Наш прогноз = {output:.5f}, прогноз Pytorch = {mynn2_prob:.5f}")

In [ ]:
# Выбираем градиентный спуск в качестве оптимизатора
mynn2_opt = torch.optim.SGD(params = mynn2.parameters(),   # параметры модели
                            lr     = lr)                   # скорость обучения

In [ ]:
# Устанавливаем обучающий режим модели
mynn2.train()

# Считаем прогнозы нейросети
mynn2_prob_train = mynn2.forward(x_train).squeeze()
mynn2_pred_train = (mynn2_prob_train >= 0.5).to(int)

# Считаем функцию потерь и точность
mynn2_loss_train = mynn2_loss(mynn2_prob_train, y_train.squeeze())

# Обнуляем посчитанные ранее градиенты функции потерь
mynn2_opt.zero_grad()

# Дифференцируем функцию потерь по весам методом обратного
# распространения ошибки (backpropagation)
mynn2_loss_train.backward()

# Посмотрим на величину производной
print(f" Наша производная = {d_log_loss_d_w111:.5f}," +
      f" Производная Pytorch = {mynn2.lincomb_1.weight.grad[0, 0]:.5f}")

In [ ]:
# Совершаем шаг алгоритма численной оптимизации (обновляем веса)
mynn2_opt.step()
# если запускать код несколько раз, то PyTorch будет
# каждый раз обновлять веса, из-за чего его результат
# может не сойтись с нашим

# Сравним обновленные веса
print(f" Наш обновленный вес = {w111_new:.5f}," +
      f" Вес обновленный Pytorch = {mynn2.lincomb_1.weight.data[0, 0]:.5f}")

Задачи для самоподготовки 🐻

1.   По аналогии рассчитайте градиент по параетрам $\omega_{122}$ и $\omega_{212}$ и сопоставьте полученный результат с `PyTorch`. 🥉

2. Не используя `PyTorch` самостоятельно запрограммируйте обучение данной нейросети: один слой с двумя нейронами и сигмоидная функция активации. 🥈

3. Включите в свою реализацию нейросети возможность использовать произвольное число признаков во входном слое и нейронов в скрытом слое. 🥈

4. Добавьте метод исключения (drouput) к своей реализации нейросети. 🥈

5. Добавьте к своей реализации нейросети возможность включать произвольное число слоев. 🥇

**Дополнительный пример ручного расчета с матрицами** 🐱

Данные

In [ ]:
# Матрица признаков
x  = np.array([[1, 2],
               [3, 4],
               [5, 6]
              ])

# Вектор значений целевой переменной
y = np.array([1, 0, 1])

Параметры (веса) нейросети. Для простоты допустим отсутствие константы (смещения).

In [ ]:
# Веса первого слоя
w1 = np.array([[-0.1, 0.2],      # веса первой линейной комбинации (первый нейрон)
               [0.3, -0.4],      # веса второй линейной комбинации (второй нейрон)
               [-0.5, 0.6]       # веса третей линейной комбинации (третий нейрон)
             ])

# Веса второго слоя
w2 = np.array([[0.7, 0.8, 0.9],  #  веса первой линейной комбинации (первый нейрон)
               [1  , 1.1, 1.2]   # веса второй линейной комбинации (второй нейрон)
             ])

# Веса выходного слоя
w3 = np.array([[1.3, -1.4]])

Функции активации и функция потерь

In [ ]:
# Функция активации первого слоя
def Tanh(x):
  val1 = np.exp(x)
  val2 = np.exp(-x)
  val = (val1 - val2) / (val1 + val2)
  return val

# Функция активации второго слоя
def ReLU(x):
  val = np.maximum(0, x)
  return val

  # Функция выходного слоя
def sigmoid(x):
  val = 1 / (1 + np.exp(-x))
  return val

# Логистическая функция потерь
def log_loss(y, o):
  o = o.ravel()
  val = np.mean(-y * np.log(o) - (1 - y) * np.log(1 - o))
  return val

Значения первого скрытого слоя

In [ ]:
# Первый слой
h1 = Tanh(np.matmul(x, w1.transpose()))
print(np.round(h1, 2))

Значения второго скрытого слоя

In [ ]:
# Второй слой
h2 = ReLU(np.matmul(h1, w2.transpose()))
print(np.round(h2, 2))

Значения выходного слоя

In [ ]:
# Выходной слой
o = sigmoid(np.matmul(h2, w3.transpose()))
print(np.round(o, 2))

Значение функции потерь

In [ ]:
# Функция потерь
loss = log_loss(y, o)
print(np.round(loss, 2))

Сопоставление результатов с PyTorch

In [ ]:
# Наша нейросеть
class MyNN3(nn.Module):
    # Конструктор
    def __init__(self, n_features, n_layer1, n_layer2):
        super().__init__()

        # Первый скрытый слой
        self.lincomb_1 = nn.Linear(in_features = n_features,
                                   out_features = n_layer1,
                                   bias = False)
        self.activation_1 = nn.Tanh()

        # Второй скрытый слой
        self.lincomb_2 = nn.Linear(in_features = n_layer1,
                                     out_features = n_layer2,
                                     bias = False)
        self.activation_2 = nn.ReLU()

        # Выходной слой
        self.lincomb_out = nn.Linear(in_features = n_layer2,
                                     out_features = 1,
                                     bias = False)
        self.activation_out = nn.Sigmoid()

    # Метод для получения прогнозов, где x отражает информацию
    # входного слоя, то есть признаки
    def forward(self, x):
        # Применяем функцию активации к линейной комбинации
        val = self.activation_1(self.lincomb_1(x))
        val = self.activation_2(self.lincomb_2(val))
        val = self.activation_out(self.lincomb_out(val))
        return val

In [ ]:
# Подготовим данные
x_train = torch.tensor(x.astype(np.float32)).to('cpu')
y_train = torch.from_numpy(y.astype(np.float32)).to('cpu')
print(x_train)
print(y_train)

In [ ]:
# Создадим модель как экземпляр нашего класса
mynn3 = MyNN3(n_features = x_train.size(dim = 1),
              n_layer1   = np.shape(w1)[0],
              n_layer2   = np.shape(w2)[0]).to('cpu')

In [ ]:
# Выбираем логистическую функцию потерь
mynn3_loss = nn.BCELoss()

In [ ]:
# Установим веса
mynn3.lincomb_1.weight.data   = torch.tensor(w1).type(x_train.dtype)
mynn3.lincomb_2.weight.data   = torch.tensor(w2).type(x_train.dtype)
mynn3.lincomb_out.weight.data = torch.tensor(w3).type(x_train.dtype)

# Посмотрим на веса, помещенные в модель
print(mynn3.lincomb_1.weight.data)
print(mynn3.lincomb_2.weight.data)
print(mynn3.lincomb_out.weight.data)

In [ ]:
# Сопоставим прогнозы (значения выходного слоя)
mynn3_prob_train = mynn3.forward(x_train).squeeze()
print("Наш прогноз = \n", o)
print("Прогноз PyTorch = \n", mynn3_prob_train)

In [ ]:
# Сопоставим значения функции потерь
mynn3_loss_train = mynn3_loss(mynn3_prob_train, y_train.squeeze())
print("Наше значение функции потерь = \n", loss)
print("Значение функции потерь PyTorch = \n", mynn3_loss_train)

Алгоритм обратного распространения ошибки (backpropagation) также реализуется достаточно просто (но громоздко), для чего могут быть полезны формулы матричного дифференцирования, описанные в [википедии](https://en.wikipedia.org/wiki/Matrix_calculus) и популярной [книге](https://www.math.uwaterloo.ca/~hwolkowi/matrixcookbook.pdf).